# Plant Disease Detection - ResNet50 Training Demo

**Simplified training notebook for quick demo**

This notebook trains ResNet50 on the plant disease dataset.
For full ablation studies, see the complete training notebook.

**Estimated Training Time:** ~2 hours on Colab Pro GPU

## 1. Setup Environment

In [ ]:
# Check if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

In [ ]:
# Install dependencies (Colab only)
if IN_COLAB:
    !pip install -q albumentations opencv-python-headless

In [ ]:
import sys
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

# Add src to path
if IN_COLAB:
    sys.path.append('/content/drive/MyDrive/plant_disease_project/src')
    BASE_PATH = '/content/drive/MyDrive/plant_disease_project'
else:
    sys.path.append('../src')
    BASE_PATH = Path('/Users/prathamr/Documents/plant-disease-detection')

# Import our modules
from data.dataset import PlantDiseaseDataset
from data.dataloader import get_train_dataloader, get_val_dataloader
from data.transforms import get_train_transforms, get_val_transforms
from models.resnet50 import get_resnet50
from training.losses import get_weighted_loss
from training.early_stopping import EarlyStopping
from training.trainer import Trainer

print("✓ All imports successful")

In [ ]:
# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ No GPU detected. Training will be slow!")

## 2. Configuration

In [ ]:
# Paths
TRAIN_CSV = os.path.join(BASE_PATH, 'data/processed/train.csv')
VAL_CSV = os.path.join(BASE_PATH, 'data/processed/val.csv')
CLASS_MAPPING = os.path.join(BASE_PATH, 'data/processed/class_mapping.json')
CHECKPOINT_DIR = os.path.join(BASE_PATH, 'models/resnet50')
TENSORBOARD_DIR = os.path.join(BASE_PATH, 'tensorboard_logs/resnet50')

# Create directories
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TENSORBOARD_DIR, exist_ok=True)

# Hyperparameters
NUM_CLASSES = 39
BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 0.001
LEARNING_RATE_FINE_TUNE = 0.0001

print("Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")

## 3. Load Data

In [ ]:
# Load class mapping
with open(CLASS_MAPPING, 'r') as f:
    class_mapping = json.load(f)

print(f"Number of classes: {class_mapping['num_classes']}")
print(f"Sample classes: {class_mapping['class_names'][:5]}")

In [ ]:
# Create datasets
print("Creating datasets...")

train_transform = get_train_transforms(image_size=224)
val_transform = get_val_transforms(image_size=224)

train_dataset = PlantDiseaseDataset(TRAIN_CSV, transform=train_transform)
val_dataset = PlantDiseaseDataset(VAL_CSV, transform=val_transform)

print(f"✓ Train dataset: {len(train_dataset)} images")
print(f"✓ Val dataset: {len(val_dataset)} images")

In [ ]:
# Create dataloaders
train_loader = get_train_dataloader(train_dataset, batch_size=BATCH_SIZE)
val_loader = get_val_dataloader(val_dataset, batch_size=BATCH_SIZE)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")

## 4. Create Model

In [ ]:
# Create ResNet50 model
model = get_resnet50(num_classes=NUM_CLASSES, pretrained=True)
model = model.to(device)

print(f"\nModel: ResNet50")
print(f"Total parameters: {model.get_num_parameters(trainable_only=False):,}")
print(f"Trainable parameters: {model.get_num_parameters(trainable_only=True):,}")

## 5. Setup Training (Phase 1: Feature Extraction)

**Phase 1 Strategy:**
- Freeze ResNet50 backbone (pretrained weights)
- Train only the final classifier layer
- Fast convergence (5 epochs)
- Higher learning rate (0.001)

In [ ]:
# Freeze backbone for Phase 1
model.freeze_backbone()

# Get class distribution for weighted loss
class_counts = train_dataset.get_class_distribution()
criterion = get_weighted_loss(class_counts, NUM_CLASSES, device)

# Optimizer (only classifier parameters)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

# Early stopping
early_stopping = EarlyStopping(patience=7, min_delta=0.001)

print("\n✓ Phase 1 setup complete")

## 6. Train Phase 1 (Feature Extraction)

In [ ]:
# Create trainer for Phase 1
trainer_phase1 = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    checkpoint_dir=os.path.join(CHECKPOINT_DIR, 'phase1'),
    tensorboard_dir=os.path.join(TENSORBOARD_DIR, 'phase1'),
    early_stopping=early_stopping,
    max_epochs=5  # Only 5 epochs for Phase 1
)

# Train
print("\n" + "="*60)
print("PHASE 1: FEATURE EXTRACTION (Backbone Frozen)")
print("="*60)
history_phase1 = trainer_phase1.fit()

## 7. Setup Training (Phase 2: Fine-Tuning)

**Phase 2 Strategy:**
- Unfreeze all layers
- Fine-tune entire model
- Lower learning rate (0.0001)
- More epochs (15-20)

In [ ]:
# Unfreeze backbone for Phase 2
model.unfreeze_backbone()

# New optimizer with lower learning rate
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE_FINE_TUNE)

# New scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

# Reset early stopping
early_stopping.reset()

print("\n✓ Phase 2 setup complete")
print(f"  Trainable parameters: {model.get_num_parameters(trainable_only=True):,}")

## 8. Train Phase 2 (Fine-Tuning)

In [ ]:
# Create trainer for Phase 2
trainer_phase2 = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    checkpoint_dir=os.path.join(CHECKPOINT_DIR, 'phase2'),
    tensorboard_dir=os.path.join(TENSORBOARD_DIR, 'phase2'),
    early_stopping=early_stopping,
    max_epochs=20  # 20 epochs for Phase 2
)

# Train
print("\n" + "="*60)
print("PHASE 2: FINE-TUNING (All Layers Trainable)")
print("="*60)
history_phase2 = trainer_phase2.fit()

## 9. Save Final Model

In [ ]:
# Copy best model from Phase 2 to main models directory
import shutil

best_model_path = os.path.join(CHECKPOINT_DIR, 'phase2/best_model.pth')
final_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

shutil.copy(best_model_path, final_model_path)

print(f"\n✓ Final model saved to: {final_model_path}")
print(f"  Best val loss: {trainer_phase2.best_val_loss:.4f}")
print(f"  Best val acc: {trainer_phase2.best_val_acc:.2f}%")

## 10. Training Complete!

### Next Steps:
1. **Evaluate on test set** → Run `03_model_evaluation_viz.ipynb`
2. **View TensorBoard logs**:
   ```bash
   tensorboard --logdir=tensorboard_logs/resnet50
   ```
3. **Use model in Streamlit app** → `streamlit run src/app/main.py`

### Files Created:
- ✅ `models/resnet50/best_model.pth` - Best model checkpoint
- ✅ `tensorboard_logs/resnet50/` - Training logs
- ✅ Phase 1 and Phase 2 checkpoints

In [ ]:
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"\nFinal Results:")
print(f"  Best Validation Loss: {trainer_phase2.best_val_loss:.4f}")
print(f"  Best Validation Accuracy: {trainer_phase2.best_val_acc:.2f}%")
print(f"\nModel saved to: {final_model_path}")
print("\nReady for evaluation and deployment! 🚀")